# 3.6 MCP Preview — Tool Integration with Model Context Protocol

**Week 4 — Agentic AI & Multi-Agent Systems**

> **This section is a conceptual preview only.** Server implementation, testing, and multi-server
> composition are covered end-to-end in Week 5. The goal here is to finish Week 4 understanding
> *what* MCP is and *why* it exists, so Week 5 can go straight into building.

## Learning objectives
- Explain the problem MCP solves and the "USB-C for AI" analogy
- Compare MCP against direct API calls / hand-rolled agent tools
- Name and describe the three MCP primitives: Tools, Resources, Prompts
- Describe MCP's architecture: Host Application, MCP Client, Protocol Layer, MCP Server
- Observe (via a simulation) the tool-discovery and execution flow, without writing any server code


## 1. What Is MCP? — "USB-C for AI"

In 3.2 you gave an agent tools by writing Python functions directly inside your agent code
(`fx_convert`, `web_search_stub`, ...). This works, but it doesn't scale: every new AI application that
wants to use "your company's ticketing system" has to **re-implement the same integration from
scratch**, in whatever framework it happens to use.

**Model Context Protocol (MCP)** standardises this: instead of hardcoding a tool inside every agent,
you build **one MCP server** that exposes the ticketing system's capabilities in a standard format.
*Any* MCP-compatible AI client (Claude Desktop, VS Code, a custom agent) can then connect to that one
server and discover/use its tools — no bespoke integration code per client.

This is why MCP is often described as **"USB-C for AI"**: one universal connector shape, instead of a
different proprietary cable for every device.


## 2. The Problem MCP Solves

Without a standard protocol:

```
N agent frameworks  x  M internal systems  =  N x M bespoke integrations
```

Every framework (LangChain, AutoGen, CrewAI, a custom agent) that wants to talk to every system
(a CRM, a ticketing tool, a database, a filesystem) needs its own glue code. That glue code is
fragile — a change to the CRM's API breaks every integration separately, and there's no shared
place to enforce security or access-scoping rules.

With MCP:

```
N agent frameworks  x  1 MCP client-server contract  x  M MCP servers  =  N + M integration points
```

You write **one server per system**, and it works with **every** MCP-compatible client, forever
(as long as the server implements the protocol).


In [ ]:
def bespoke_integration_count(n_frameworks: int, m_systems: int) -> int:
    return n_frameworks * m_systems

def mcp_integration_count(n_frameworks: int, m_systems: int) -> int:
    # each system needs one server; each framework needs to speak the protocol once
    return n_frameworks + m_systems

print(f"{'Frameworks':<12}{'Systems':<10}{'Bespoke':<10}{'MCP':<6}")
for n, m in [(2, 3), (4, 6), (6, 12)]:
    print(f"{n:<12}{m:<10}{bespoke_integration_count(n, m):<10}{mcp_integration_count(n, m):<6}")


The gap between the "Bespoke" and "MCP" columns widens fast as your organisation adopts more
frameworks and more internal systems — which is exactly the industry problem MCP was designed to fix.


## 3. MCP vs. Direct API Calls and Agents

| | Direct API calls (3.2 style) | Agent framework tools (LangChain/AutoGen) | MCP |
|---|---|---|---|
| Where tool logic lives | Inside your agent code | Inside your agent code, framework-specific format | Inside an independent **server**, protocol-agnostic |
| Reuse across frameworks | None | Requires rewriting for each framework's tool format | Any MCP-compatible client can use it unchanged |
| Who executes the tool | Your process | Your process | The MCP server's process (possibly a different machine) |
| Access control | Ad hoc, per integration | Ad hoc, per integration | First-class: servers can scope exactly what's exposed |

MCP doesn't replace tool-calling (the ReAct loop from 3.1 is unchanged) — it **relocates and
standardises where the tool's implementation lives**, and how a client discovers what's available.


## 4. The Three MCP Primitives

- **Tools** — actions the AI can *execute*, often with side effects (e.g. "create a GitHub issue",
  "run this SQL query", "send this email").
- **Resources** — *read-only* data the server can provide to enrich the AI's context (e.g. "the
  contents of this file", "today's on-call schedule", "an employee directory entry").
- **Prompts** — *reusable instruction templates* the server provides, so a client can request a
  pre-crafted, tested prompt instead of each client re-inventing its own phrasing.

Think of it as: **Tools = verbs, Resources = nouns, Prompts = sentence templates.**


In [ ]:
from dataclasses import dataclass
from typing import Callable, Dict, Any, List

@dataclass
class MCPTool:
    name: str
    description: str
    handler: Callable[..., Any]

@dataclass
class MCPResource:
    uri: str
    description: str
    fetcher: Callable[[], Any]

@dataclass
class MCPPrompt:
    name: str
    template: str

# A tiny mock MCP server for a fictional "ticketing" system -- illustrative only.
# Week 5 builds a real one of these using the FastMCP library and @mcp.tool() / @mcp.resource() decorators.
class MockMCPServer:
    def __init__(self, name: str):
        self.name = name
        self.tools: Dict[str, MCPTool] = {}
        self.resources: Dict[str, MCPResource] = {}
        self.prompts: Dict[str, MCPPrompt] = {}

    def register_tool(self, tool: MCPTool):
        self.tools[tool.name] = tool

    def register_resource(self, resource: MCPResource):
        self.resources[resource.uri] = resource

    def register_prompt(self, prompt: MCPPrompt):
        self.prompts[prompt.name] = prompt

    # --- Protocol-shaped discovery methods (mirrors ListToolsRequest / ListResourcesRequest) ---
    def list_tools(self) -> List[Dict[str, str]]:
        return [{"name": t.name, "description": t.description} for t in self.tools.values()]

    def list_resources(self) -> List[Dict[str, str]]:
        return [{"uri": r.uri, "description": r.description} for r in self.resources.values()]

    def call_tool(self, name: str, **kwargs) -> Any:
        return self.tools[name].handler(**kwargs)

    def read_resource(self, uri: str) -> Any:
        return self.resources[uri].fetcher()

server = MockMCPServer("ticketing-mcp-server")

server.register_tool(MCPTool(
    name="create_ticket",
    description="Create a support ticket. Args: title (str), priority (str)",
    handler=lambda title, priority: f"Created ticket '{title}' with priority={priority} (id=TCK-1042)"
))
server.register_resource(MCPResource(
    uri="ticketing://oncall-schedule/today",
    description="Who is on call today",
    fetcher=lambda: {"oncall": "Ananya Rao", "shift": "09:00-17:00 IST"}
))
server.register_prompt(MCPPrompt(
    name="ticket_summary_prompt",
    template="Summarise the following ticket thread for a status update in 2 sentences: {thread}"
))

print("Server:", server.name)
print("Tools:", server.list_tools())
print("Resources:", server.list_resources())
print("Prompts:", list(server.prompts.keys()))


## 5. MCP Architecture Overview

```
+-----------------+        +-------------+        +------------------+        +---------------+
| Host Application| <----> | MCP Client  | <----> | Protocol Layer   | <----> | MCP Server     |
| (Claude Desktop,|        | (embedded in|        | (JSON-RPC-style  |        | (your ticketing|
|  VS Code, ...)  |        |  the host)  |        |  message format) |        |  integration)  |
+-----------------+        +-------------+        +------------------+        +---------------+
```

- **Host Application** — the AI-facing app the user actually interacts with.
- **MCP Client** — lives inside the host; speaks the protocol on the host's behalf.
- **Protocol Layer** — the standardised message format/transport (stdio for local tools,
  streamable HTTP for remote servers — covered in depth in Week 5, Section 4.1).
- **MCP Server** — the independent process that owns the actual tools/resources/prompts and
  executes them.

The request flow for a single tool call is: the host discovers available tools
(`ListToolsRequest`) → the model decides to use one → the client sends `CallToolRequest` → the
server executes it → a structured result flows back to the host. This is, again, the exact same
ReAct loop from 3.1 — MCP just standardises the wire format and moves execution into an
independent server process.


In [ ]:
# Simulate the end-to-end discovery -> selection -> execution flow a Host Application would drive.

def host_application_turn(server: MockMCPServer, user_goal: str):
    print(f"USER GOAL: {user_goal}\n")

    # Step 1: discovery (ListToolsRequest equivalent)
    available = server.list_tools()
    print("Step 1 - Host discovers available tools:")
    for t in available:
        print(f"   - {t['name']}: {t['description']}")

    # Step 2: the model (brain) decides which tool fits -- mocked here as a simple keyword match
    print("\nStep 2 - Model reasons about the goal and selects a tool...")
    if "ticket" in user_goal.lower():
        chosen_tool, args = "create_ticket", {"title": "Login page returns 500 error", "priority": "high"}
    else:
        chosen_tool, args = None, {}

    if chosen_tool is None:
        print("   -> no suitable tool found.")
        return

    print(f"   -> selected tool: {chosen_tool} with args {args}")

    # Step 3: CallToolRequest equivalent
    print("\nStep 3 - Client sends CallToolRequest to the MCP server...")
    result = server.call_tool(chosen_tool, **args)

    # Step 4: structured result flows back to the host
    print(f"\nStep 4 - Server executes and returns a structured result:\n   {result}")

host_application_turn(server, "Please open a ticket for the login page 500 error, it's urgent.")


### Where MCP fits in a multi-agent system

Any agent from Sections 3.3–3.5 (a Researcher, a Support Quality Analyst, an Escalator...) can be
given access to real systems by connecting it to one or more MCP servers, instead of hand-writing
Python tool functions as we did in 3.2. The agent's reasoning loop doesn't change at all — only
*where the tool implementation lives and how it's discovered* changes.


## Key Takeaways

- MCP is a **standard protocol** ("USB-C for AI") that lets any compatible AI client use any
  compatible server's tools — solving the N-frameworks × M-systems integration explosion.
- The three primitives are **Tools** (actions), **Resources** (read-only data), and **Prompts**
  (reusable templates).
- The architecture is **Host Application → MCP Client → Protocol Layer → MCP Server**, and the
  request flow (`ListToolsRequest` → `CallToolRequest` → structured result) is the same ReAct loop
  from 3.1, just standardised and moved to an independent server process.
- **This was a preview only.** Week 5 builds real MCP servers with the FastMCP library, adds
  security/governance/guardrails, and composes multi-server ecosystems — starting directly from
  these concepts with no re-introduction.

## Check your understanding
1. What specific integration problem does MCP solve that hand-written agent tools (3.2) don't?
2. Match each MCP primitive (Tools, Resources, Prompts) to its rough equivalent: "a verb", "a noun",
   "a sentence template."
3. In the architecture diagram, which component actually executes a tool call — the Host, the Client,
   or the Server?

---
*End of Week 4 preview notebooks (Sections 3.1–3.6). Full server-building content continues in
Week 5 — Model Context Protocol: Build, Compose & Deploy.*
